# ChildGuard AI v4 — Colab eğitimi (A100)

Hücreleri sırayla çalıştırın. **Duman testi `✅ SMOKE OK` demeden tam eğitime geçmeyin.** Ayrıntılar: `docs/COLAB.md`.

## Adım 0 — Çalışma zamanı

**Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** (varsa *Yüksek RAM* açık).

## Adım 1 — Donanım kontrolü

In [ ]:
import torch, sys, shutil, psutil
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "GPU yok! Çalışma zamanı türünü A100 yap."
props = torch.cuda.get_device_properties(0)
print("GPU    :", props.name, f"{props.total_memory/2**30:.0f} GB", "| bf16:", torch.cuda.is_bf16_supported())
print("RAM    :", round(psutil.virtual_memory().total/2**30), "GB")
print("Disk   :", round(shutil.disk_usage("/content").free/2**30), "GB boş")

## Adım 2 — Drive

In [ ]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/ChildGuard_v4"
for sub in ["checkpoints", "models", "reports", "hf_cache"]:
    os.makedirs(f"{WORK}/{sub}", exist_ok=True)
os.environ["HF_HOME"] = f"{WORK}/hf_cache"
os.environ["CG_WORK"] = WORK                      # veri, model ve rapor Drive'a
os.environ["CG_CKPT"] = "/content/cg_checkpoints" # checkpoint'ler hızlı yerel diske (her biri ~4 GB)
print(WORK, os.listdir(WORK))
du = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive: {du.free/2**30:.0f} GB boş / {du.total/2**30:.0f} GB")

## Adım 3 — Kütüphaneler

In [ ]:
!pip install -q -U "transformers>=4.44" "datasets>=2.20" accelerate emoji pyarrow scikit-learn matplotlib pytest
import transformers, datasets, sklearn
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| sklearn", sklearn.__version__)

## Adım 4 — Kodu çek

Büyük LFS dosyalarından yalnızca karşılaştırma için gerekenler indirilir (~0,5 GB).

In [ ]:
!which git-lfs > /dev/null || (apt-get -qq update && apt-get -qq install -y git-lfs > /dev/null)
%cd /content
!rm -rf ChildGuardHybrid
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/omerozcn/ChildGuardHybrid.git
%cd /content/ChildGuardHybrid
!git lfs install --local
!git lfs pull --include="final_models/bert_general/*,data/raw/*"
!git log --oneline -1 && ls

## Adım 5 — Birim testleri (~2 dk)

Kodun bu ortamda (Python/kütüphane sürümleri) doğru çalıştığını, eğitime hiç başlamadan doğrular.

Beklenen: `51 passed`. Bir test kırılırsa çıktıyı paylaşın; eğitime geçmeyin.

In [ ]:
!python -m pytest -q tests -p no:cacheprovider

## Adım 6 — Duman testi (~5–10 dk)

Bütün hattı gerçek veriyle ve gerçek modelle (A100'de `roberta-large`), çok küçük ölçekte çalıştırır:
bellek ön testi, bf16, veri indirme, sözcüksel model, transformer, kalibrasyon, değerlendirme, v3 karşılaştırması, model yükleme.
Ayrı bir klasöre (`$WORK/smoke`) yazar, asıl eğitime karışmaz.

Son satırda **`✅ SMOKE OK`** görmelisiniz. Görmüyorsanız tam eğitime geçmeyin.

In [ ]:
!python -u -m training.run_all --smoke

## Adım 7 — Tam eğitim (~1–2 saat)

Duman testinde ölçülen hıza göre tahmini süreyi başta yazar. Oturum düşerse **aynı hücreyi tekrar çalıştırın**:
tamamlanan aşamalar (veri, sözcüksel model — Drive'da) atlanır. Checkpoint'ler yerel diskte olduğu için
oturum *tamamen* sıfırlanırsa transformer baştan eğitilir; yalnızca hücre/çekirdek yeniden başlarsa kaldığı yerden devam eder.

In [ ]:
!python -u -m training.run_all --profile full

## Adım 8 — Sonuçlar

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{WORK}/reports/eval_report.md", encoding="utf-8").read()))
for f in ["pr_curves.png", "gold_recall_by_category.png"]:
    p = f"{WORK}/reports/{f}"
    if os.path.exists(p): display(Image(p))

## Adım 9 — Modeli bilgisayara al

Model Drive'da `ChildGuard_v4/models/v4` klasöründe. İsterseniz zip olarak da indirin:

Bilgisayarınızda zip'teki `models/v4` klasörünü depodaki `final_models/v4` olarak koyun; backend onu yükler.

In [ ]:
!cd "$WORK" && zip -qr /content/childguard_v4.zip models/v4 reports
from google.colab import files; files.download("/content/childguard_v4.zip")